# Set Piece Analytics

Este notebook extrai eventos de bolas paradas do StatsBomb Open Data.

**Saída:**
- `data/<competicao>/<temporada>.parquet` — um arquivo por competição/temporada
- `data/set_pieces.parquet` — arquivo centralizado com tudo concatenado

**Execute as células em ordem.** As células 1–4 são de exploração e não gravam nada. O processamento começa na célula 5.

---

## Célula 1 — Imports e configurações

In [ ]:
import os
import re
import warnings
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from statsbombpy import sb
from tqdm import tqdm

warnings.filterwarnings("ignore", module="statsbombpy.*")

OUTPUT_DIR = "data"
CENTRAL_FILE = os.path.join(OUTPUT_DIR, "set_pieces.parquet")
MAX_WORKERS = 16

os.makedirs(OUTPUT_DIR, exist_ok=True)


def slugify(text):
    """Converte nome de competição/temporada para nome de pasta/arquivo seguro."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


print("Ambiente pronto.")
print(f"Exemplos de slugify: 'La Liga' → '{slugify('La Liga')}', '2017/2018' → '{slugify('2017/2018')}'")


## Célula 2 — Explorar competições disponíveis

In [ ]:
competitions = sb.competitions()
print(f"Total de competições/temporadas: {len(competitions)}")
competitions[["competition_id", "competition_name", "season_id", "season_name"]].head(5)

## Célula 3 — Teste com uma única partida

Antes de processar tudo, validamos a estrutura dos dados e as colunas disponíveis.

In [ ]:
# La Liga 2017/18: competition_id=11, season_id=1
sample_matches = sb.matches(competition_id=11, season_id=1)
sample_match_id = int(sample_matches["match_id"].iloc[0])
print(f"Partida de teste: {sample_match_id}")

sample_events = sb.events(match_id=sample_match_id)
print(f"Total de eventos: {len(sample_events)}")
print(f"Colunas disponíveis:")
print(sample_events.columns.tolist())

In [ ]:
# Verificar set pieces presentes na partida de teste
passes = sample_events[sample_events["type"] == "Pass"]
shots = sample_events[sample_events["type"] == "Shot"]

print("=== Tipos de Pass ===")
print(passes["pass_type"].value_counts())

print("\n=== Tipos de Shot ===")
print(shots["shot_type"].value_counts())

print("\n=== Colunas de shot ===")
shot_cols = [c for c in sample_events.columns if c.startswith("shot")]
print(shot_cols)

## Célula 4 — Funções de extração

Cada função recebe o DataFrame de eventos de uma partida e retorna uma lista de dicionários, um por oportunidade de bola parada.

In [ ]:
MINUTE_BINS = [0, 15, 30, 45, 60, 75, 91]
MINUTE_LABELS = ["0-15", "16-30", "31-45", "46-60", "61-75", "76-90+"]

OUTPUT_COLUMNS = [
    "match_id", "competition_name", "season_name", "team_name", "set_piece_type",
    "period", "minute", "minute_band", "shots_generated", "goals_scored",
    "xg_sum", "origin_x", "origin_y", "shot_x", "shot_y",
]


def get_minute_band(minute):
    for i, upper in enumerate(MINUTE_BINS[1:]):
        if minute <= upper:
            return MINUTE_LABELS[i]
    return "76-90+"


def _extract_xy_columns(df, col="location"):
    pairs = df[col].apply(lambda loc: loc[:2] if isinstance(loc, list) and len(loc) >= 2 else [None, None])
    xy = pd.DataFrame(pairs.tolist(), index=df.index, columns=["_x", "_y"])
    return xy["_x"], xy["_y"]


def extract_pass_based(events_df, pass_type_value, set_piece_label, match_meta):
    trigger_passes = events_df[
        (events_df["type"] == "Pass") &
        (events_df["pass_type"] == pass_type_value)
    ].copy()

    if trigger_passes.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    shots = events_df[events_df["type"] == "Shot"].copy()

    if not shots.empty:
        shots["_is_goal"] = (shots["shot_outcome"] == "Goal").astype(int)
        shots["_xg"] = shots["shot_statsbomb_xg"].fillna(0.0)
        shots["_shot_x"], shots["_shot_y"] = _extract_xy_columns(shots)

        first_shot = shots.groupby("possession").first()[["_shot_x", "_shot_y"]]
        poss_agg = shots.groupby("possession").agg(
            shots_generated=("_is_goal", "count"),
            goals_scored=("_is_goal", "sum"),
            xg_sum=("_xg", "sum"),
        ).join(first_shot)
    else:
        poss_agg = pd.DataFrame(
            columns=["shots_generated", "goals_scored", "xg_sum", "_shot_x", "_shot_y"]
        )

    trigger_passes = trigger_passes.join(poss_agg, on="possession", how="left")
    trigger_passes["shots_generated"] = trigger_passes["shots_generated"].fillna(0).astype(int)
    trigger_passes["goals_scored"] = trigger_passes["goals_scored"].fillna(0).astype(int)
    trigger_passes["xg_sum"] = trigger_passes["xg_sum"].fillna(0.0)

    origin_x, origin_y = _extract_xy_columns(trigger_passes)

    out = pd.DataFrame({
        "match_id": match_meta["match_id"],
        "competition_name": match_meta["competition_name"],
        "season_name": match_meta["season_name"],
        "team_name": trigger_passes["team"].values,
        "set_piece_type": set_piece_label,
        "period": trigger_passes["period"].values,
        "minute": trigger_passes["minute"].values,
        "minute_band": trigger_passes["minute"].apply(get_minute_band).values,
        "shots_generated": trigger_passes["shots_generated"].values,
        "goals_scored": trigger_passes["goals_scored"].values,
        "xg_sum": trigger_passes["xg_sum"].values,
        "origin_x": origin_x.values,
        "origin_y": origin_y.values,
        "shot_x": trigger_passes["_shot_x"].values if "_shot_x" in trigger_passes.columns else None,
        "shot_y": trigger_passes["_shot_y"].values if "_shot_y" in trigger_passes.columns else None,
    })
    return out


def extract_shot_based(events_df, shot_type_value, set_piece_label, match_meta):
    trigger_shots = events_df[
        (events_df["type"] == "Shot") &
        (events_df["shot_type"] == shot_type_value)
    ].copy()

    if trigger_shots.empty:
        return pd.DataFrame(columns=OUTPUT_COLUMNS)

    origin_x, origin_y = _extract_xy_columns(trigger_shots)

    out = pd.DataFrame({
        "match_id": match_meta["match_id"],
        "competition_name": match_meta["competition_name"],
        "season_name": match_meta["season_name"],
        "team_name": trigger_shots["team"].values,
        "set_piece_type": set_piece_label,
        "period": trigger_shots["period"].values,
        "minute": trigger_shots["minute"].values,
        "minute_band": trigger_shots["minute"].apply(get_minute_band).values,
        "shots_generated": 1,
        "goals_scored": (trigger_shots["shot_outcome"] == "Goal").astype(int).values,
        "xg_sum": trigger_shots["shot_statsbomb_xg"].fillna(0.0).values,
        "origin_x": origin_x.values,
        "origin_y": origin_y.values,
        "shot_x": origin_x.values,
        "shot_y": origin_y.values,
    })
    return out


def extract_set_pieces(events_df, match_meta):
    frames = [
        extract_pass_based(events_df, "Corner",    "Escanteio",      match_meta),
        extract_pass_based(events_df, "Free Kick", "Falta Indireta", match_meta),
        extract_pass_based(events_df, "Throw-in",  "Lateral",        match_meta),
        extract_shot_based(events_df, "Free Kick", "Falta Direta",   match_meta),
        extract_shot_based(events_df, "Penalty",   "Penalti",        match_meta),
    ]
    non_empty = [f for f in frames if not f.empty]
    return pd.concat(non_empty, ignore_index=True) if non_empty else pd.DataFrame(columns=OUTPUT_COLUMNS)


# --- Teste das funções com a partida de amostra ---
sample_match_row = sample_matches.iloc[0]
sample_meta = {
    "match_id": int(sample_match_row["match_id"]),
    "competition_name": sample_match_row["competition"],
    "season_name": sample_match_row["season"],
}

test_df = extract_set_pieces(sample_events, sample_meta)

print(f"Set pieces extraídos da partida de teste: {len(test_df)}")
print(test_df["set_piece_type"].value_counts())
test_df.head()


## Célula 5 — Loop completo

Processa todas as competições e temporadas. Cada combinação é salva em:
`data/<competicao>/<temporada>.parquet`

Se o loop for interrompido, os arquivos já gerados são preservados e podem ser reaproveitados na célula 7.

> **Atenção:** requer conexão com internet. Para ~4.235 partidas pode levar vários minutos.

In [ ]:
def _fetch_and_extract(match_id, comp_name, season_name):
    match_meta = {
        "match_id": match_id,
        "competition_name": comp_name,
        "season_name": season_name,
    }
    events_df = sb.events(match_id=match_id)
    return extract_set_pieces(events_df, match_meta)


errors = []
skipped = []
total_records = 0

for _, comp_row in tqdm(competitions.iterrows(), total=len(competitions), desc="Competições"):
    comp_id = int(comp_row["competition_id"])
    season_id = int(comp_row["season_id"])
    comp_name = comp_row["competition_name"]
    season_name = comp_row["season_name"]

    comp_slug = slugify(comp_name)
    season_slug = slugify(season_name)
    comp_dir = os.path.join(OUTPUT_DIR, comp_slug)
    season_file = os.path.join(comp_dir, f"{season_slug}.parquet")

    # Pular se já foi processado (permite retomar de onde parou)
    if os.path.exists(season_file):
        skipped.append(f"{comp_name} / {season_name}")
        continue

    os.makedirs(comp_dir, exist_ok=True)

    try:
        matches_df = sb.matches(competition_id=comp_id, season_id=season_id)
    except Exception as e:
        errors.append({"where": f"matches({comp_name} / {season_name})", "error": str(e)})
        continue

    print(f"\n  {comp_name} / {season_name} — {len(matches_df)} partidas")

    season_frames = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_id = {
            executor.submit(_fetch_and_extract, int(row["match_id"]), comp_name, season_name): int(row["match_id"])
            for _, row in matches_df.iterrows()
        }
        for future in tqdm(as_completed(future_to_id), total=len(future_to_id), desc="  Partidas", leave=False):
            match_id = future_to_id[future]
            try:
                season_frames.append(future.result())
            except Exception as e:
                errors.append({"where": f"events(match_id={match_id})", "error": str(e)})

    non_empty_frames = [f for f in season_frames if not f.empty]
    if non_empty_frames:
        season_df = pd.concat(non_empty_frames, ignore_index=True)
        season_df.to_parquet(season_file, index=False)
        total_records += len(season_df)
        print(f"  → {len(season_df)} bolas paradas | Total acumulado: {total_records:,}")

print(f"\nConcluído. Total de registros: {total_records:,}")
if skipped:
    print(f"Já existiam e foram pulados: {len(skipped)}")
if errors:
    print(f"Erros encontrados: {len(errors)}")
    for err in errors[:5]:
        print(f"  - {err}")


## Célula 6 — Validação

Lê todos os parquets individuais e valida antes de gerar o arquivo centralizado.

In [ ]:
# Encontrar todos os parquets individuais gerados
individual_files = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in files:
        path = os.path.join(root, f)
        # Excluir o centralizado (se já existir)
        if f.endswith(".parquet") and path != CENTRAL_FILE:
            individual_files.append(path)

print(f"Arquivos individuais encontrados: {len(individual_files)}")

# Carregar e concatenar para validação
df = pd.concat([pd.read_parquet(f) for f in individual_files], ignore_index=True)

print(f"\n=== Shape total ===")
print(df.shape)

print("\n=== Set pieces por tipo ===")
print(df["set_piece_type"].value_counts())

print("\n=== Competições presentes ===")
print(df["competition_name"].value_counts())

print("\n=== Nulos por coluna ===")
print(df.isnull().sum())

print("\n=== xG quando há shot (deve ser > 0 na maioria) ===")
has_shot = df[df["shots_generated"] == 1]
print(f"Linhas com shot: {len(has_shot)}")
print(f"xg_sum = 0 quando há shot: {(has_shot['xg_sum'] == 0).sum()}")

df.head()

## Célula 7 — Gerar arquivo centralizado

Concatena todos os parquets individuais em `data/set_pieces.parquet`.

In [ ]:
df.to_parquet(CENTRAL_FILE, index=False)
file_size_mb = os.path.getsize(CENTRAL_FILE) / (1024 * 1024)
print(f"Salvo em: {CENTRAL_FILE}")
print(f"Tamanho: {file_size_mb:.2f} MB")
print(f"Linhas: {len(df):,}")

## Célula 8 — Verificações finais

Confirma que o arquivo centralizado pode ser lido e calcula os KPIs globais.

In [ ]:
df_check = pd.read_parquet(CENTRAL_FILE)

print("=== Leitura do parquet centralizado OK ===")
print(f"Shape: {df_check.shape}")
print(f"Colunas: {df_check.columns.tolist()}")

print("\n=== KPIs globais ===")
total = len(df_check)
tf = df_check["shots_generated"].sum() / total
tc = df_check["goals_scored"].sum() / total
xg_medio = df_check["xg_sum"].sum() / total

print(f"Total de bolas paradas: {total:,}")
print(f"Taxa de Finalização (TF): {tf:.2%}")
print(f"Taxa de Conversão (TC): {tc:.2%}")
print(f"xG Médio: {xg_medio:.4f}")

print("\n=== Estrutura gerada em data_processed/ ===")
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    print(f"{indent}{folder}/")
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{indent}  {f}  ({size_kb:.0f} KB)")